# 06 · Your first model: `nn.Module`, loss functions and optimizers

**Objective:** To turn automatic differentiation (notebook 05) into a training procedure: fit a model to data by gradient descent, first by hand and then with `torch.nn` and `torch.optim`.

## Part 1: Concepts (90 mins)

### 1. The learning problem
We have data pairs $(x_i, y_i)$, $i = 1, \dots, N$, and a model $f_\theta(x)$ with parameters $\theta$. *Training* means choosing the $\theta$ that minimizes a **loss function**, for example the mean squared error

$$L(\theta) = \frac{1}{N} \sum_{i=1}^{N} \big(f_\theta(x_i) - y_i\big)^2 .$$

**Gradient descent** repeats one simple update, with a small step size $\eta$ called the *learning rate*:

$$\theta \leftarrow \theta - \eta \, \nabla_\theta L(\theta).$$

`autograd` computes $\nabla_\theta L$ for us. We start with the simplest model, a straight line $f(x) = w x + b$.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)  # reproducible noise

true_w, true_b = 2.0, -1.0
X = torch.linspace(-1, 1, 100).unsqueeze(1)   # shape (100, 1): 100 samples, 1 feature
y = true_w * X + true_b + 0.2 * torch.randn(X.shape)

print(X.shape, y.shape)
plt.scatter(X, y, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### 2. Gradient descent by hand
Three details matter:
- the parameter update is done inside `torch.no_grad()`, because the update itself must not become part of the computational graph;
- PyTorch **accumulates** gradients in `.grad`, so they must be reset to zero after every update;
- `loss.item()` converts a one-element tensor into a Python number (useful for logging).

In [ ]:
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
losses = []

for step in range(200):
    y_pred = X * w + b                   # forward pass
    loss = ((y_pred - y) ** 2).mean()    # mean squared error
    loss.backward()                      # compute dL/dw and dL/db

    with torch.no_grad():                # update the parameters
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                       # reset the gradients
    b.grad.zero_()
    losses.append(loss.item())

print(f"w = {w.item():.3f} (true {true_w}), b = {b.item():.3f} (true {true_b})")

plt.plot(losses)
plt.xlabel("step")
plt.ylabel("MSE loss")
plt.yscale("log")
plt.show()

### 3. The same model with `torch.nn` and `torch.optim`
Writing every parameter and update by hand does not scale. PyTorch provides three building blocks:

- **`nn.Module`**: a class that holds the parameters and defines the `forward` computation;
- **loss functions** in `torch.nn`, e.g. `nn.MSELoss` for regression, `nn.CrossEntropyLoss` for classification;
- **optimizers** in `torch.optim`, e.g. `SGD` (the update above) or `Adam`.

In [ ]:
import torch.nn as nn

class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)  # holds w and b

    def forward(self, x):
        return self.linear(x)

model = LinearRegression()
print(model)
for name, p in model.named_parameters():
    print(name, tuple(p.shape), "requires_grad =", p.requires_grad)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(200):
    y_pred = model(X)             # forward pass
    loss = loss_fn(y_pred, y)     # compute loss
    optimizer.zero_grad()         # reset gradients
    loss.backward()               # backward pass
    optimizer.step()              # update parameters

print(f"w = {model.linear.weight.item():.3f}, b = {model.linear.bias.item():.3f}")

The correspondence with the manual version:

| By hand | With `torch.nn` / `torch.optim` |
|---|---|
| `X * w + b` | `model(X)` |
| `((y_pred - y) ** 2).mean()` | `loss_fn(y_pred, y)` |
| `w.grad.zero_()`, `b.grad.zero_()` | `optimizer.zero_grad()` |
| `w -= lr * w.grad` (inside `no_grad`) | `optimizer.step()` |

These five lines (forward, loss, zero_grad, backward, step) are the core of every training loop you will write.
Here each step uses the whole dataset (*full-batch* gradient descent); notebooks 07 and 08 split the data into mini-batches.

### 4. Beyond straight lines: a small neural network
A linear model cannot fit a curved relationship. A **multi-layer perceptron (MLP)** alternates linear layers with a non-linear *activation function*.
Without the activation, a stack of linear layers is still a single linear map.

`nn.Sequential` builds a model by chaining layers, without writing a class.

In [ ]:
torch.manual_seed(0)
X_curve = torch.linspace(-2, 2, 200).unsqueeze(1)
y_curve = torch.sin(3 * X_curve) + 0.1 * torch.randn(X_curve.shape)

def train(model, X, y, optimizer, n_steps=2000):
    # Full-batch training with MSE; returns the list of losses.
    loss_fn = nn.MSELoss()
    losses = []
    for step in range(n_steps):
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(1)
linear_model = nn.Linear(1, 1)
mlp = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
)
print(mlp)
print("Number of parameters:", sum(p.numel() for p in mlp.parameters()))

losses_linear = train(linear_model, X_curve, y_curve, torch.optim.Adam(linear_model.parameters(), lr=0.01))
losses_mlp = train(mlp, X_curve, y_curve, torch.optim.Adam(mlp.parameters(), lr=0.01))
print(f"final loss: linear {losses_linear[-1]:.4f}, MLP {losses_mlp[-1]:.4f}")

In [ ]:
with torch.no_grad():
    fit_linear = linear_model(X_curve)
    fit_mlp = mlp(X_curve)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(X_curve, y_curve, s=8, alpha=0.5, label="data")
ax1.plot(X_curve, fit_linear, "r", label="linear")
ax1.plot(X_curve, fit_mlp, "k", label="MLP")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend()

ax2.plot(losses_linear, label="linear")
ax2.plot(losses_mlp, label="MLP")
ax2.set_yscale("log")
ax2.set_xlabel("step")
ax2.set_ylabel("MSE loss")
ax2.legend()
plt.tight_layout()
plt.show()

**A word of caution.** A low *training* loss only says that the model fits the data it has seen. Whether it works on new data must be measured on data held out from training: this is the subject of notebook 09.

## Part 2: Exercises & Debugging (90 mins)

### Lab 6.1: The learning rate
* **Task:** Train the `LinearRegression` model on `(X, y)` with `SGD` for 100 steps, for each learning rate in `[0.001, 0.01, 0.1, 0.5, 1.2]`. Store the loss curves and plot them on the same axes (log scale on the y axis).
* **Questions:** Which learning rates are too small? What happens with `lr=1.2`?
* **Hint:** call `torch.manual_seed(0)` before creating each model, so that all runs start from the same initial parameters.

In [ ]:
# Your code here


### Lab 6.2: Choosing the optimizer
* **Task:** Fit the curve data `(X_curve, y_curve)` with the same MLP architecture, once with `SGD(lr=0.1)` and once with `Adam(lr=0.01)`, for 2000 steps each. Start both from the same initialization and compare the loss curves.

In [ ]:
# Your code here


### Lab 6.3: How big should the network be?
* **Task:** Train MLPs with hidden sizes `[2, 8, 32, 128]` on the curve data (Adam, `lr=0.01`, 2000 steps) and plot the four fitted curves on top of the data. Print the number of parameters of each model.
* **Question:** The training loss keeps decreasing as the model grows. Does that mean the largest model is the best one? What would you need to answer this question?

In [ ]:
# Your code here


### Debugging Focus 1: Forgetting `zero_grad()`
- **The Problem:** The loss behaves erratically or diverges, even with a reasonable learning rate.
- **The Cause:** `backward()` *adds* the new gradient to whatever is already stored in `.grad`. Without `optimizer.zero_grad()`, each step uses the sum of all past gradients.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss = 4.5 * w ** 2        # dloss/dw = 9 w = 9 at w = 1
    loss.backward()
    print(f"after backward #{i + 1}: w.grad = {w.grad.item()}")   # 9, 18, 27: accumulated!

### Debugging Focus 2: Silent broadcasting between predictions and targets
- **The Problem:** Training runs without errors, but the model does not learn.
- **The Cause:** The model outputs shape `(N, 1)`, while the targets have shape `(N,)`. Subtracting them broadcasts to an `(N, N)` matrix, so the loss compares every prediction with every target.
- **The Solution:** Always print the shapes of predictions and targets. Keep them identical, e.g. with `y.unsqueeze(1)` or `pred.squeeze(1)`. `nn.MSELoss` prints a `UserWarning` in this situation: do not ignore warnings.

In [ ]:
model = LinearRegression()
pred = model(X)             # shape (100, 1)
y_flat = y.squeeze(1)       # shape (100,)
print("pred:", pred.shape, " target:", y_flat.shape, " difference:", (pred - y_flat).shape)
loss = nn.MSELoss()(pred, y_flat)   # look at the warning below